# Evaluacija retrieval sistema

Metrike kvaliteta rangiranja se računaju za 3 sistema:

1. gte_full - GTE embedding kompletnog dokumenta proizvoda
2. e5_maxp - E5 chunkovi sa maksimalnim chunk score-om po proizvodu
3. e5_meanchunks - prosečan E5 embedding svih chunkova proizvoda

Za svaki upit računaju se nDCG@5, MRR@5 i HitRate@5 , nakon čega se rezultati proseče na nivou sistema.

In [2]:
%pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.5 MB/s eta 0:00:00


## 1. Priprema okruženja

In [5]:
import sys
import torch
from torchmetrics.functional.retrieval import retrieval_normalized_dcg,retrieval_reciprocal_rank,retrieval_hit_rate
import pandas as pd
import numpy as np
from pathlib import Path

if torch.cuda.is_available():
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/RecSys')
else:
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT.resolve())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/RecSys


## 2. Putanja projekta i učitavanje evaluacionih artefakata

- all_rankings.csv - upit, identifikator proizvoda, njegovu poziciju i oznaku sistema koji je vratio tu poziciju za upit.
- qrels.csv - ocene relevantnosti za parove (proizvod , upit)

In [6]:
EVAL_DIR = PROJECT_ROOT / 'data' / 'evaluation'
RANKINGS_PATH = EVAL_DIR / 'all_rankings.csv'
QRELS_PATH = EVAL_DIR / 'qrels.csv'
DEPTH = 5

rankings = pd.read_csv(RANKINGS_PATH)
qrels = pd.read_csv(QRELS_PATH)

## 3. Računanje metrika po sistemu i upitu

In [ ]:
records = []

for candidate_id in rankings.candidate_id.unique():
    candidate_rankings = rankings.loc[rankings.candidate_id == candidate_id]

    for query_id, query_qrels in qrels.groupby("query_id"):
        query_ranking = candidate_rankings.loc[candidate_rankings.query_id == query_id,["logical_product_id", "rank"]]
        evaluation_data = query_qrels.merge(query_ranking,on="logical_product_id",how="left")

        predicted_scores = torch.tensor(np.where(evaluation_data["rank"].notna(),DEPTH- evaluation_data["rank"].fillna(0)+ 1,0,),dtype=torch.float32)
        relevance = torch.tensor(evaluation_data["relevance"].to_numpy(),dtype=torch.long)
        ndcg_target = (2 ** relevance) - 1
        binary_target = (relevance >= 2).long()

        ndcg = retrieval_normalized_dcg(predicted_scores,ndcg_target,top_k=DEPTH)
        mrr = retrieval_reciprocal_rank(predicted_scores,binary_target,top_k=DEPTH)
        hit_rate = retrieval_hit_rate(predicted_scores,binary_target,top_k=DEPTH)

        records.append({"candidate_id": candidate_id,"query_id": query_id,"ndcg@5": float(ndcg),"mrr@5": float(mrr),"hitrate@5": float(hit_rate),})

per_query_metrics = pd.DataFrame(records)
summary_metrics = (per_query_metrics.groupby("candidate_id", as_index=False)[["ndcg@5", "mrr@5", "hitrate@5"]].mean())

display(summary_metrics)

,candidate_id,ndcg@5,mrr@5,hitrate@5
0,e5_maxp,0.692995,0.770486,0.916667
1,e5_meanchunks,0.703390,0.768056,0.937500
2,gte_full,0.742955,0.832292,0.916667


In [ ]:
per_query_metrics.to_csv(EVAL_DIR / "metrics_per_query.csv",index=False)
summary_metrics.to_csv(EVAL_DIR / "metrics_summary.csv",index=False)

### 4.1. Rezultati razvojne evaluacije

Prema primarnoj metrici nDCG@5, **gte_full** ostvaruje najbolji rezultat.Takođe ima i najveći MRR@5, dok e5_mean ostvaruje najbolji HitRate@5.

In [ ]:

def wait_for_backend(url: str, timeout: int = 30):
    """Polls the backend API health endpoint until it is ready."""
    start_time = time.time()

    with st.spinner("⏳ Waiting for backend API to initialize..."):
        while True:
            try:
                # Send a lightweight request to the health check endpoint
                response = requests.get("{FASTAPI_URL}/health", timeout=2)
                if response.status_code == 200:
                    break  
            except requests.exceptions.RequestException:
                # Backend is still starting up or down
                pass

            # Timeout mechanism to prevent infinite loops
            if time.time() - start_time > timeout:
                st.error("❌ The backend API failed to start in time.")
                st.stop()  

            time.sleep(1)  


# 1. Run the health check first
wait_for_backend(FASTAPI_URL)

In [ ]:
flex = st.container(horizontal=True)
flex.markdown(
    """
    <style>.card {
        padding: 20px;
        border-radius: 10px;
        border: 1px solid #e6e6e6;
        background-color: #f9f9f9;
        margin-bottom: 10px;
    }
    </style>
    """,
    unsafe_allow_html=True,
)
flex.markdown(
    """
    <style>.card {
        padding: 20px;
        border-radius: 10px;
        border: 1px solid #e6e6e6;
        background-color: #f9f9f9;
        margin-bottom: 10px;
    }
    </style>
    """,
    unsafe_allow_html=True,
)


In [ ]:
import streamlit.components.v1 as components

# bootstrap 4 collapse example
components.html(
    """
    <link rel="stylesheet" href="https://maxcdn.bootstrapcdn.com/bootstrap/4.0.0/css/bootstrap.min.css" integrity="sha384-Gn5384xqQ1aoWXA+058RXPxPg6fy4IWvTNh0E263XmFcJlSAwiGgFAW/dAiS6JXm" crossorigin="anonymous">
    <script src="https://code.jquery.com/jquery-3.2.1.slim.min.js" integrity="sha384-KJ3o2DKtIkvYIK3UENzmM7KCkRr/rE9/Qpg6aAZGJwFDMVNA/GpGFF93hXpG5KkN" crossorigin="anonymous"></script>
    <script src="https://maxcdn.bootstrapcdn.com/bootstrap/4.0.0/js/bootstrap.min.js" integrity="sha384-JZR6Spejh4U02d8jOt6vLEHfe/JQGiRRSQQxSfFWpi1MquVdAyjUar5+76PVCmYl" crossorigin="anonymous"></script>
    <div id="accordion">
      <div class="card">
        <div class="card-header" id="headingOne">
          <h5 class="mb-0">
            <button class="btn btn-link" data-toggle="collapse" data-target="#collapseOne" aria-expanded="true" aria-controls="collapseOne">
            Collapsible Group Item #1
            </button>
          </h5>
        </div>
        <div id="collapseOne" class="collapse show" aria-labelledby="headingOne" data-parent="#accordion">
          <div class="card-body">
            Collapsible Group Item #1 content
          </div>
        </div>
      </div>
      <div class="card">
        <div class="card-header" id="headingTwo">
          <h5 class="mb-0">
            <button class="btn btn-link collapsed" data-toggle="collapse" data-target="#collapseTwo" aria-expanded="false" aria-controls="collapseTwo">
            Collapsible Group Item #2
            </button>
          </h5>
        </div>
        <div id="collapseTwo" class="collapse" aria-labelledby="headingTwo" data-parent="#accordion">
          <div class="card-body">
            Collapsible Group Item #2 content
          </div>
        </div>
      </div>
    </div>
    """,
    height=600,
)
